### Create a classificator model based on LLM-generated features and TF-IDF terms for the hate speech dataset

In [69]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from utils.utils import preprocessing, process_txt_files

In [70]:
# Load the dataset
from datasets import load_dataset

data_load = load_dataset("banking77")

In [71]:
df_llm_features = process_txt_files('../../data/outputs/bank77', 'bank77')
df_llm_features.drop(columns=['id', 'custom_id'], inplace=True)
df_llm_features = pd.get_dummies(
    df_llm_features, sparse=False, prefix_sep='_'
)

In [72]:
df_train = pd.DataFrame(data_load['train'])
df_test = pd.DataFrame(data_load['test'])

df_train_text = df_train[['text', 'label']]
df_test_text = df_test[['text', 'label']]
X_train_text = df_train_text['text'].apply(lambda x: preprocessing(x))
y_train_text = df_train_text['label']

X_test_text = df_test_text['text'].apply(lambda x: preprocessing(x))
y_test_text = df_test_text['label']

In [73]:
X_text = df_train_text.drop(columns=["label"])
y = df_train_text["label"]
X_train_llm_features = df_llm_features[:10003]
X_test_llm_features = df_llm_features[10003:]

In [79]:
def tf_idf(train, test):
    vectorizer = TfidfVectorizer()
    train_tfidf = vectorizer.fit_transform(train)
    test_tfdidf = vectorizer.transform(test)
    return pd.DataFrame(train_tfidf.toarray(), columns=vectorizer.get_feature_names_out()),  pd.DataFrame(test_tfdidf.toarray(), columns=vectorizer.get_feature_names_out())

In [80]:
X_train_text, X_test_text = tf_idf(X_train_text, X_test_text)

In [81]:
# Merge the LLM features with the tf-idf features
X_train = pd.concat((X_train_text.reset_index(drop=True), X_train_llm_features.reset_index(drop=True)), axis=1)
X_test = pd.concat((X_test_text.reset_index(drop=True), X_test_llm_features.reset_index(drop=True)), axis=1)

In [82]:
# # Train a classifier
clf = GradientBoostingClassifier(random_state=42)
clf.fit(X_train, y_train_text)

GradientBoostingClassifier(random_state=42)

In [83]:
# Test the classifier
y_pred = clf.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test_text, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95        40
           1       0.93      1.00      0.96        40
           2       1.00      0.97      0.99        40
           3       0.94      0.75      0.83        40
           4       0.94      0.80      0.86        40
           5       0.46      0.82      0.59        40
           6       0.86      0.95      0.90        40
           7       0.78      0.88      0.82        40
           8       0.94      0.85      0.89        40
           9       1.00      0.93      0.96        40
          10       0.75      0.60      0.67        40
          11       0.46      0.78      0.58        40
          12       0.85      0.72      0.78        40
          13       0.93      0.97      0.95        40
          14       0.72      0.78      0.75        40
          15       0.54      0.68      0.60        40
          16       0.40      0.68      0.50        40
          17       0.83    